## Уточнение постановки задачи

Изначально проект формулировался как сервис для парсинга ВКС-встреч: получение транскрипта, извлечение полезной информации и дальнейшая загрузка результатов в базу знаний или использование для формирования мемо.

После первичного анализа стало понятно, что полная задача парсинга ВКС состоит из нескольких подзадач:

1. распознавание речи;
2. диаризация участников;
3. очистка и нормализация транскрипта;
4. исправление ошибок в корпоративных сущностях;
5. извлечение решений, поручений и фактов;
6. структурирование результата для базы знаний или корпоративного ассистента.

В рамках текущего исследования я фокусируюсь не на всём end-to-end пайплайне, а на отдельной критичной подзадаче: исправлении ошибок ASR в корпоративных сущностях.

Такое сужение задачи связано с тем, что именно искажение корпоративных названий часто делает транскрипт непригодным для дальнейшего автоматического анализа. Например, если ASR неверно распознаёт название системы, проекта, организации, документа или аббревиатуры, то последующие этапы — поиск по базе знаний, построение мемо, извлечение поручений и загрузка в корпоративный контекст — начинают работать хуже.

Поэтому текущая исследовательская задача формулируется так:

**подготовить датасет для оценки методов восстановления корректных корпоративных сущностей в ASR-транскриптах ВКС.**

Таким образом, проект по-прежнему относится к задаче парсинга ВКС, но текущая итерация посвящена одному из его ключевых компонентов: формированию словаря корпоративных сущностей и датасета ASR-подобных искажений для последующей оценки качества исправления.


# EDA

## Вытаскиваем объектоподобные блоки из сырой выгрузки

In [ ]:
from pathlib import Path
import re
import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 100)

RAW_PATH = Path("../data/raw_dataset.txt")

raw_text = RAW_PATH.read_text(encoding="utf-8")
print(f"Размер файла: {len(raw_text):,} символов")
print(raw_text[:500])

In [ ]:
def extract_json_like_objects(text: str) -> list[str]:
    """
    Достаёт блоки вида {...} из сырого txt.
    Работает терпимо даже если весь файл не является валидным JSON.
    """
    objects = []
    depth = 0
    start = None

    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1

        elif ch == "}":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    objects.append(text[start:i + 1])
                    start = None

    return objects


blocks = extract_json_like_objects(raw_text)

print(f"Найдено объектоподобных блоков: {len(blocks)}")
print(blocks[0][:500])

In [ ]:
def extract_string_field(block: str, key: str) -> str | None:
    """
    Извлекает строковое поле из блока.
    Умеет переживать неэкранированные кавычки внутри значения
    """
    pattern = (
        rf'"{re.escape(key)}"\s*:\s*"'
        rf'(?P<value>.*?)'
        rf'(?="\s*,\s*"[A-Za-zА-Яа-яЁё_][A-Za-zА-Яа-яЁё_]*"\s*:|"\s*}})'
    )

    match = re.search(pattern, block, flags=re.DOTALL)
    if not match:
        return None

    value = match.group("value")
    value = value.replace("\n", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value


def extract_list_field(block: str, key: str) -> list[str]:
    """
    Грубое извлечение списка строк.
    Если поле битое — возвращаем пустой список.
    """
    pattern = rf'"{re.escape(key)}"\s*:\s*\[(?P<value>.*?)\]'
    match = re.search(pattern, block, flags=re.DOTALL)
    if not match:
        return []

    content = match.group("value")
    items = re.findall(r'"(.*?)"', content, flags=re.DOTALL)
    items = [re.sub(r"\s+", " ", item).strip() for item in items]
    return [item for item in items if item]


def parse_block(block: str, record_id: int) -> dict:
    return {
        "record_id": record_id,
        "entity_name": extract_string_field(block, "entity_name"),
        "entity_type": extract_string_field(block, "entity_type"),
        "evidence": extract_string_field(block, "evidence"),
        "confidence": extract_string_field(block, "confidence"),
        "why_important_for_asr": extract_string_field(block, "why_important_for_asr"),
        "aliases": extract_list_field(block, "aliases"),
        "possible_asr_errors": extract_list_field(block, "possible_asr_errors"),
        "raw_block": block,
    }


records = [parse_block(block, i) for i, block in enumerate(blocks, start=1)]

df = pd.DataFrame(records)

print(df.shape)
df.head()

In [7]:
df.shape

(301, 9)

## Базовая чистка строк

In [ ]:
def normalize_spaces(value):
    if value is None:
        return None
    value = str(value)
    value = value.replace("\xa0", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value or None


def normalize_quotes(value):
    if value is None:
        return None

    value = str(value)

    replacements = {
        "“": "«",
        "”": "»",
        "„": "«",
        "‟": "»",
        "’": "'",
        "‘": "'",
    }

    for old, new in replacements.items():
        value = value.replace(old, new)

    value = normalize_spaces(value)
    return value


for col in ["entity_name", "entity_type", "evidence", "confidence", "why_important_for_asr"]:
    df[col] = df[col].apply(normalize_quotes)

df["entity_type"] = df["entity_type"].str.upper()

df.head()

In [ ]:
def make_entity_key(name: str | None) -> str | None:
    if name is None:
        return None

    value = name.lower()
    value = value.replace("ё", "е")

    # убираем кавычки и лишнюю пунктуацию для сравнения дублей
    value = re.sub(r"[«»\"'`“”]", "", value)
    value = re.sub(r"\s+", " ", value)
    value = value.strip()

    return value or None


df["entity_key"] = df["entity_name"].apply(make_entity_key)

df[["entity_name", "entity_type", "entity_key", "evidence"]].head(10)

## Флаги качества извлекаемых данных

In [ ]:
ALLOWED_TYPES = {
    "SYSTEM",
    "PROJECT",
    "DEPARTMENT",
    "ORGANIZATION",
    "ABBREVIATION",
    "DOCUMENT",
    "DOMAIN_TERM",
    "TRANSPORT_OBJECT",
    "INFRA_OBJECT",
}

GENERIC_EVIDENCE_PATTERNS = [
    "в базе есть документ",
    "в транскриптах встречается",
    "в транскриптах часто упоминается",
    "в контексте есть ссылка",
    "внутренняя система",
    "внешняя проектная организация",
    "типовой документ",
    "подразделение, отвечающее",
]


def is_generic_evidence(evidence: str | None) -> bool:
    if not evidence:
        return True

    e = evidence.lower()
    return any(pattern in e for pattern in GENERIC_EVIDENCE_PATTERNS)


def looks_like_person(name: str | None, entity_type: str | None) -> bool:
    if not name:
        return False

    if entity_type == "PERSON":
        return True

    # Фамилия И. О.
    if re.search(r"\b[А-ЯЁ][а-яё]+ [А-ЯЁ]\.\s?[А-ЯЁ]\.", name):
        return True

    person_markers = [
        "мэр москвы",
        "заместитель мэра",
    ]

    return name.lower() in person_markers


def looks_like_metric_or_sentence(name: str | None) -> bool:
    if not name:
        return True

    n = name.strip()

    # просто число
    if re.fullmatch(r"\d+", n):
        return True

    # длинное утверждение
    if len(n.split()) >= 7:
        return True

    # начинается с числа и похоже на факт/метрику
    if re.match(r"^\d", n) and len(n.split()) >= 3:
        return True

    metric_markers = [
        "%",
        "млрд",
        "млн",
        "тысяч",
        "количество",
        "доля",
        "интервал",
        "парность",
        "протяженность",
        "поездок",
        "рабочих мест",
        "стали доступнее",
    ]

    n_lower = n.lower()
    return any(marker in n_lower for marker in metric_markers)


def looks_like_generated_series(name: str | None) -> bool:
    if not name:
        return False

    # ПТК «Сфера-1» ... ПТК «Сфера-50»
    if re.search(r"птк\s*[«\"]?сфера[-\s]?\d+", name.lower()):
        return True

    return False


def has_bad_type(entity_type: str | None) -> bool:
    if not entity_type:
        return True
    return entity_type not in ALLOWED_TYPES


df["flag_missing_name"] = df["entity_name"].isna()
df["flag_missing_evidence"] = df["evidence"].isna()
df["flag_generic_evidence"] = df["evidence"].apply(is_generic_evidence)
df["flag_person"] = df.apply(lambda row: looks_like_person(row["entity_name"], row["entity_type"]), axis=1)
df["flag_metric_or_sentence"] = df["entity_name"].apply(looks_like_metric_or_sentence)
df["flag_generated_series"] = df["entity_name"].apply(looks_like_generated_series)
df["flag_bad_type"] = df["entity_type"].apply(has_bad_type)

df.head()

In [12]:
def assign_preliminary_status(row) -> str:
    if row["flag_missing_name"]:
        return "rejected"

    if row["flag_person"]:
        return "rejected"

    if row["flag_metric_or_sentence"]:
        return "rejected"

    if row["flag_bad_type"]:
        return "needs_verification"

    if row["flag_generated_series"]:
        return "needs_verification"

    if row["flag_missing_evidence"] or row["flag_generic_evidence"]:
        return "needs_verification"

    return "candidate"


df["preliminary_status"] = df.apply(assign_preliminary_status, axis=1)

df["preliminary_status"].value_counts()

preliminary_status
candidate             199
needs_verification     73
rejected               29
Name: count, dtype: int64

In [13]:
# Сначала сортируем так, чтобы candidate был выше needs_verification и rejected
status_rank = {
    "candidate": 0,
    "needs_verification": 1,
    "rejected": 2,
}

df["status_rank"] = df["preliminary_status"].map(status_rank).fillna(99)

df_sorted = df.sort_values(
    by=["entity_key", "status_rank", "record_id"],
    ascending=[True, True, True],
)

df_dedup = df_sorted.drop_duplicates(subset=["entity_key"], keep="first").copy()

print("До дедупликации:", len(df))
print("После дедупликации:", len(df_dedup))

df_dedup["preliminary_status"].value_counts()

До дедупликации: 301
После дедупликации: 224


preliminary_status
candidate             141
needs_verification     63
rejected               20
Name: count, dtype: int64

In [14]:
print("Всего записей:", len(df))
print("Уникальных entity_key:", df["entity_key"].nunique())
print("После дедупликации:", len(df_dedup))

display(df["entity_type"].value_counts(dropna=False).rename_axis("entity_type").reset_index(name="count"))

display(df["preliminary_status"].value_counts().rename_axis("status").reset_index(name="count"))

Всего записей: 301
Уникальных entity_key: 223
После дедупликации: 224


,entity_type,count
0,SYSTEM,128
1,DOMAIN_TERM,55
2,PROJECT,50
3,DEPARTMENT,24
4,ORGANIZATION,21
5,DOCUMENT,14
6,ABBREVIATION,6
7,PERSON,2
8,"DOMAIN_TERM"", ""EVIDENCE: ""КЛЮЧЕВЫЕ_ТРАНСПОРТ_ДТ",1


,status,count
0,candidate,199
1,needs_verification,73
2,rejected,29


In [ ]:
# Примеры кандидатов
candidate_cols = [
    "entity_name",
    "entity_type",
    "evidence",
    "preliminary_status",
    "flag_generic_evidence",
    "flag_metric_or_sentence",
    "flag_person",
    "flag_generated_series",
]

display(df_dedup[df_dedup["preliminary_status"] == "candidate"][candidate_cols].head(5))

In [ ]:
# Примеры того, что надо проверить
display(df_dedup[df_dedup["preliminary_status"] == "needs_verification"][candidate_cols].head(5))

In [ ]:
# Примеры отклонённых
display(df_dedup[df_dedup["preliminary_status"] == "rejected"][candidate_cols].head(5))

## Сохранение результатов

In [22]:
OUT_DIR = Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Полная распарсенная таблица
df.to_csv(OUT_DIR / "raw_candidates_parsed.csv", index=False, encoding="utf-8-sig")

# Дедуплицированная таблица
df_dedup.to_csv(OUT_DIR / "raw_candidates_deduplicated.csv", index=False, encoding="utf-8-sig")

# Предварительно хорошие кандидаты
df_candidates = df_dedup[df_dedup["preliminary_status"] == "candidate"].copy()
df_candidates.to_csv(OUT_DIR / "candidate_entities.csv", index=False, encoding="utf-8-sig")

# Требуют проверки
df_needs_verification = df_dedup[df_dedup["preliminary_status"] == "needs_verification"].copy()
df_needs_verification.to_csv(OUT_DIR / "needs_verification_entities.csv", index=False, encoding="utf-8-sig")

# Отклонённые
df_rejected = df_dedup[df_dedup["preliminary_status"] == "rejected"].copy()
df_rejected.to_csv(OUT_DIR / "rejected_entities.csv", index=False, encoding="utf-8-sig")

print("Saved to:", OUT_DIR.resolve())

Saved to: /Users/nikita/Documents/MLOps ITMO/mfdp-meeting-memo/homework_04/outputs


In [30]:
n_total = len(df)
n_unique = df["entity_key"].nunique()
n_dedup = len(df_dedup)
status_counts = df_dedup["preliminary_status"].value_counts().to_dict()
type_counts = df_dedup["entity_type"].value_counts(dropna=False).to_dict()

print("Выводы:")
print(f"- Сырая выгрузка содержит {n_total} объектоподобных записей.")
print(f"- После нормализации найдено {n_unique} уникальных ключей сущностей.")
print(f"- После дедупликации осталось {n_dedup} записей.")
print(f"- Кандидатов без явных проблем: {status_counts.get('candidate', 0)}.")
print(f"- Требуют ручной проверки: {status_counts.get('needs_verification', 0)}.")
print(f"- Предварительно отклонены: {status_counts.get('rejected', 0)}.")
print("- Основные проблемы: дубли, неверные типы, персоналии, числовые показатели, общие evidence и возможные галлюцинации.")
print("- Выгрузка пригодна как raw candidate set, но не как финальный эталонный датасет.",
        "\n- Плюс надо написать скрипт, который портит корпоративные сущности, затем уже начинать работу над фичей исправления.")

Выводы:
- Сырая выгрузка содержит 301 объектоподобных записей.
- После нормализации найдено 223 уникальных ключей сущностей.
- После дедупликации осталось 224 записей.
- Кандидатов без явных проблем: 141.
- Требуют ручной проверки: 63.
- Предварительно отклонены: 20.
- Основные проблемы: дубли, неверные типы, персоналии, числовые показатели, общие evidence и возможные галлюцинации.
- Выгрузка пригодна как raw candidate set, но не как финальный эталонный датасет. 
- Плюс надо написать скрипт, который портит корпоративные сущности, затем уже начинать работу над фичей исправления.
